# Compare several variants with a shared control

[All How-to Guides](README.md)

Compare two product variants with a shared control and adjust for the family of two hypotheses.

## 1. Set the assignments and one outcome

In the sample data, `0` identifies control. Labels `1` and `2` identify the variants. Define the comparison family before examining the results.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from hypex import ABTest
from hypex.dataset import Dataset, TargetRole, TreatmentRole

rng = np.random.default_rng(42)
group = rng.permutation(np.repeat([0, 1, 2], 200))
frame = pd.DataFrame(
    {"group": group, "metric": rng.normal(100, 10, size=len(group)) + group * 2.5}
)
data = Dataset(
    roles={"group": TreatmentRole(), "metric": TargetRole()},
    data=frame,
)
frame["group"].value_counts().sort_index()

group
0    200
1    200
2    200
Name: count, dtype: int64

The control and each variant contain 200 observations.

## 2. Estimate both effects

Use the Bonferroni adjustment. `summary` shows the effect estimate and interval for each comparison with control.

HypEx recognizes A/B/n from the labels in the `TreatmentRole` column. `multitest_method="bonferroni"` selects the adjustment.

In the full `summary`, find groups `1` and `2` and read `difference`, `ci lower`, and `ci upper`.

In [2]:
bonferroni = ABTest(multitest_method="bonferroni").execute(data)
bonferroni.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,metric,1,99.49154,101.598357,2.106817,2.117584,0.102214,4.11142,OK,4.005811e-02
1,metric,2,99.49154,105.235759,5.744219,5.773576,3.819679,7.66876,OK,1.026667e-08


The rows show variant 1 minus control and variant 2 minus control. The intervals in `summary` are for the individual comparisons.

## 3. Read the adjusted p-values

Compare Bonferroni and Holm on the same data. Here, Bonferroni multiplies each original p-value by two and caps it at one. Holm uses the order of the original p-values.

`multitest_method="holm"` is the `ABTest` default. The example spells it out for the comparison with Bonferroni.

In [3]:
holm = ABTest(multitest_method="holm").execute(data)
print("Bonferroni")
display(bonferroni.multitest)
print("Holm")
display(holm.multitest)

Bonferroni


,field,test,old p-value,new p-value,correction,rejected,group
0,metric,TTest,0.04005810593677801,0.08011621187355603,0.5,False,1
1,metric,TTest,1.0266674933545377e-08,2.0533349867090755e-08,0.5,True,2


Holm


,field,test,old p-value,new p-value,correction,rejected,group
0,metric,TTest,0.04005810593677801,0.04005810593677801,1.0,True,1
1,metric,TTest,1.0266674933545377e-08,2.0533349867090755e-08,0.5,True,2


`new p-value` and `rejected` reflect the adjustment for each variant. Groups 1 and 2 each share the same control. If you need to compare variants 1 and 2 directly, include that comparison in the analysis plan.

For variant 1, the adjusted p-value is about 0.0801 with Bonferroni and 0.0401 with Holm. At a significance level of 0.05, the decision depends on the adjustment. Specify it in the analysis plan.

For two metrics and two groups, see the [Holm adjustment guide](multiple-testing.ipynb).